In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Loading Dataset
df = pd.read_csv("/content/drive/MyDrive/Credit Card Fraud /creditcard.csv")
df = df.sort_values(by='Time').reset_index(drop=True)
df = df.iloc[:50000]

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df['Class'].value_counts()

In [ ]:
# Preprocessing
# Handling Imbalance
# separating the classes
X = df.drop('Class', axis=1)
y = df['Class']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Creating Sequences
def create_sequences(X, y, time_steps=20):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 10

X_seq, y_seq = create_sequences(X_scaled, y.values, TIME_STEPS)

print(X_seq.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq,
    test_size=0.2,
    random_state=42,
    stratify = y_seq)

In [ ]:
# Class Weights
weights = class_weight.compute_class_weight(
    class_weight = 'balanced',
    classes = np.unique(y_train),
    y = y_train
)

class_weights = {0 : weights[0], 1: weights[1]}

In [ ]:
# Building LSTM model
model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.3))

model.add(LSTM(32))
model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid'))

In [ ]:
# compile the model
model.compile(
    loss = 'binary_crossentropy',
    optimizer = 'adam',
    metrics = ['accuracy']
)

In [ ]:
model.summary()

In [ ]:
# Training the model with early stopping
early_stop = EarlyStopping(
    monitor = 'val_loss',
    patience = 3,
    restore_best_weights = True
)
history = model.fit(
    X_train, y_train,
    epochs = 10,
    batch_size = 64,
    validation_data = (X_test, y_test),
    callbacks = [early_stop],
    class_weight = class_weights
)

In [ ]:
y_prob = model.predict(X_test).ravel() # flatten predictions

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

best_f1 = -1
best_threshold = 0.5 # Default threshold

for t in thresholds:
    y_pred = (y_prob > t).astype(int)

    print(f"\n🔹 Threshold: {t}")

    # Handle zero division safely
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

    # Update best_threshold if current F1-score is better
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

    # Debug info (VERY IMPORTANT)
    print("Predicted Fraud Count:", sum(y_pred))

In [ ]:
# Final Evaluation
y_pred = (y_prob > best_threshold).astype(int)

print("Best Threshold Used:", best_threshold)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression

# Reshape X_train and X_test for Logistic Regression (flatten the time steps)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Initialize and train Logistic Regression model
lr = LogisticRegression(solver='liblinear', random_state=42, class_weight=class_weights)
lr.fit(X_train_flat, y_train)

In [ ]:
# Model Comparison

lr_prob = lr.predict_proba(X_test_flat)[:, 1]

threshold = 0.3  # you can tune this

lr_pred = (lr_prob > threshold).astype(int)

print("Logistic Regression (Tuned):\n", classification_report(y_test, lr_pred))

In [ ]:
# Training Graph
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title("Loss Curve")
plt.show()

In [ ]:
print("FINAL MODEL PERFORMANCE")

print("\nLSTM Model:")
print(classification_report(y_test, y_pred))

print("\nKey Insight:")
print("Model achieves high recall, meaning it successfully detects most fraud cases.")
print("This is crucial in financial systems where missing fraud is costly.")

In [ ]:
print("Unique predictions LR:", np.unique(lr_pred))
print("Unique predictions LSTM:", np.unique(y_pred))

In [ ]:
print("LSTM Recall:", recall_score(y_test, y_pred))
print("LR Recall:", recall_score(y_test, lr_pred))

In [ ]:
# Visual Comparisons
models = ['LSTM', 'Logistic Regression']

recall_scores = [
    recall_score(y_test, y_pred),
    recall_score(y_test, lr_pred)
]

plt.bar(models, recall_scores)
plt.title("Recall Comparison")
plt.ylabel("Recall")
plt.show()

In [ ]:
print("Fraud cases in test set:", sum(y_test))

In [ ]:
print("Predicted fraud count:", sum(y_pred))

In [ ]:
model.save("fraud_lstm_model.h5")

In [ ]:
import joblib
joblib.dump(scaler, "scaler.pkl")

In [ ]:
# from google.colab import files
# files.download('scaler.pkl')

In [ ]:
import joblib

joblib.dump(model, "creditcard.pkl")

In [ ]:
import pickle

with open("creditcard.pkl", "rb") as f:
    model = pickle.load(f)